# Processing Geometry Data for Machine Learning Models
## Erica Keklak | 2025-06-25
The area of interest for this project is the lower 48 United States, which exclude Alaska and Hawaii as well as Washington DC, minor outlying islands, territories, protectorates, etc. We need a shapefile or something similar to show the boundaries we want once we can map the data.

In [16]:
# Import necessary libraries and packages

import pandas as pd
import geopandas as gpd

## Part 1/2: Counties

In [19]:
# Process county data

county_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Raw Data Backups/Geometry/cb_2024_us_county_500k/cb_2024_us_county_500k.shp')
county_gdf.head()

,STATEFP,COUNTYFP,COUNTYNS,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,STATE_NAME,LSAD,ALAND,AWATER,geometry
0,01,069,00161560,0500000US01069,01069,Houston,Houston County,AL,Alabama,06,1501742250,4795418,"POLYGON ((-85.712 31.197, -85.709 31.198, -85...."
1,01,023,00161537,0500000US01023,01023,Choctaw,Choctaw County,AL,Alabama,06,2365900084,19114321,"POLYGON ((-88.473 31.894, -88.469 31.93, -88.4..."
2,01,113,00161583,0500000US01113,01113,Russell,Russell County,AL,Alabama,06,1660653961,15562947,"POLYGON ((-85.435 32.318, -85.434 32.392, -85...."
3,10,005,00217269,0500000US10005,10005,Sussex,Sussex County,DE,Delaware,06,2424590442,674129051,"POLYGON ((-75.723 38.83, -75.615 38.834, -75.5..."
4,01,071,00161561,0500000US01071,01071,Jackson,Jackson County,AL,Alabama,06,2792044612,126334711,"MULTIPOLYGON (((-86.154 34.534, -86.15 34.534,..."


In [20]:
# Eliminate counties not of interest

county_gdf = county_gdf.drop(county_gdf[county_gdf['STATE_NAME'] == "Alaska"].index)
county_gdf = county_gdf.drop(county_gdf[county_gdf['STATE_NAME'] == "Hawaii"].index)
county_gdf = county_gdf.drop(county_gdf[county_gdf['STATE_NAME'] == "District of Columbia"].index)
county_gdf = county_gdf.drop(county_gdf[county_gdf['STATE_NAME'] == "United States Virgin Islands"].index)
county_gdf = county_gdf.drop(county_gdf[county_gdf['STATE_NAME'] == "American Samoa"].index)
county_gdf = county_gdf.drop(county_gdf[county_gdf['STATE_NAME'] == "Guam"].index)
county_gdf = county_gdf.drop(county_gdf[county_gdf['STATE_NAME'] == "Puerto Rico"].index)
county_gdf = county_gdf.drop(county_gdf[county_gdf['STATE_NAME'] == "Commonwealth of the Northern Mariana Islands"].index)

# Check

county_gdf.head()

,STATEFP,COUNTYFP,COUNTYNS,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,STATE_NAME,LSAD,ALAND,AWATER,geometry
0,01,069,00161560,0500000US01069,01069,Houston,Houston County,AL,Alabama,06,1501742250,4795418,"POLYGON ((-85.712 31.197, -85.709 31.198, -85...."
1,01,023,00161537,0500000US01023,01023,Choctaw,Choctaw County,AL,Alabama,06,2365900084,19114321,"POLYGON ((-88.473 31.894, -88.469 31.93, -88.4..."
2,01,113,00161583,0500000US01113,01113,Russell,Russell County,AL,Alabama,06,1660653961,15562947,"POLYGON ((-85.435 32.318, -85.434 32.392, -85...."
3,10,005,00217269,0500000US10005,10005,Sussex,Sussex County,DE,Delaware,06,2424590442,674129051,"POLYGON ((-75.723 38.83, -75.615 38.834, -75.5..."
4,01,071,00161561,0500000US01071,01071,Jackson,Jackson County,AL,Alabama,06,2792044612,126334711,"MULTIPOLYGON (((-86.154 34.534, -86.15 34.534,..."


In [23]:
# Streamline data for use in AI training

new_county_gdf = gpd.GeoDataFrame({})
new_county_gdf['county'] = county_gdf['NAME']
new_county_gdf['state'] = county_gdf['STATE_NAME']
new_county_gdf['land_area'] = county_gdf['ALAND']
new_county_gdf['water_area'] = county_gdf['AWATER']
new_county_gdf['geometry'] = county_gdf['geometry']
new_county_gdf.set_geometry('geometry')

new_county_gdf.to_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/county_geometry.shp')

new_county_gdf.head()

C:\Users\EK111\AppData\Local\Temp\ipykernel_55528\4273525031.py:8: FutureWarning: You are adding a column named 'geometry' to a GeoDataFrame constructed without an active geometry column. Currently, this automatically sets the active geometry column to 'geometry' but in the future that will no longer happen. Instead, either provide geometry to the GeoDataFrame constructor (GeoDataFrame(... geometry=GeoSeries()) or use `set_geometry('geometry')` to explicitly set the active geometry column.
  new_county_gdf['geometry'] = county_gdf['geometry']


,county,state,land_area,water_area,geometry
0,Houston,Alabama,1501742250,4795418,"POLYGON ((-85.712 31.197, -85.709 31.198, -85...."
1,Choctaw,Alabama,2365900084,19114321,"POLYGON ((-88.473 31.894, -88.469 31.93, -88.4..."
2,Russell,Alabama,1660653961,15562947,"POLYGON ((-85.435 32.318, -85.434 32.392, -85...."
3,Sussex,Delaware,2424590442,674129051,"POLYGON ((-75.723 38.83, -75.615 38.834, -75.5..."
4,Jackson,Alabama,2792044612,126334711,"MULTIPOLYGON (((-86.154 34.534, -86.15 34.534,..."


In [24]:
# Iterate the same geometry information for every year of data analysis except for prediction years

county_geometry_annual = gpd.GeoDataFrame({'county': [], 'state': [], 'land_area': [], 'water_area': [], 'geometry': []})
for i in range(2014, 2025): # from 2014 to 2024; doesn't count the last number 2025
    one_year = new_county_gdf
    one_year['year'] = [i] * len(new_county_gdf)
    county_geometry_annual = county_geometry_annual.merge(one_year, how = 'outer')

county_geometry_annual.head()

,county,state,land_area,water_area,geometry,year
0,Houston,Alabama,1501742250,4795418,"POLYGON ((-85.712 31.197, -85.709 31.198, -85....",2014
1,Choctaw,Alabama,2365900084,19114321,"POLYGON ((-88.473 31.894, -88.469 31.93, -88.4...",2014
2,Russell,Alabama,1660653961,15562947,"POLYGON ((-85.435 32.318, -85.434 32.392, -85....",2014
3,Sussex,Delaware,2424590442,674129051,"POLYGON ((-75.723 38.83, -75.615 38.834, -75.5...",2014
4,Jackson,Alabama,2792044612,126334711,"MULTIPOLYGON (((-86.154 34.534, -86.15 34.534,...",2014


In [39]:
# Save annual county data

county_geometry_annual = county_geometry_annual.set_geometry('geometry')
county_geometry_annual.to_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/county_geometry_annual.shp')

## Part 2/2: States

In [26]:
# Process state data

state_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Raw Data Backups/Geometry/cb_2024_us_state_500k/cb_2024_us_state_500k.shp')
state_gdf.head()

,STATEFP,STATENS,GEOIDFQ,GEOID,STUSPS,NAME,LSAD,ALAND,AWATER,geometry
0,35,00897535,0400000US35,35,NM,New Mexico,00,314198519809,726531289,"POLYGON ((-109.05 31.48, -109.05 31.5, -109.05..."
1,46,01785534,0400000US46,46,SD,South Dakota,00,196341670967,3387563375,"POLYGON ((-104.06 44.998, -104.05 44.998, -104..."
2,06,01779778,0400000US06,06,CA,California,00,403673433805,20291632828,"MULTIPOLYGON (((-118.6 33.479, -118.6 33.478, ..."
3,21,01779786,0400000US21,21,KY,Kentucky,00,102266755818,2384136185,"MULTIPOLYGON (((-89.406 36.528, -89.399 36.542..."
4,01,01779775,0400000US01,01,AL,Alabama,00,131185561946,4581813708,"MULTIPOLYGON (((-88.053 30.507, -88.051 30.509..."


In [28]:
# Eliminate states not of interest

state_gdf = state_gdf.drop(state_gdf[state_gdf['NAME'] == "Alaska"].index)
state_gdf = state_gdf.drop(state_gdf[state_gdf['NAME'] == "Hawaii"].index)
state_gdf = state_gdf.drop(state_gdf[state_gdf['NAME'] == "District of Columbia"].index)
state_gdf = state_gdf.drop(state_gdf[state_gdf['NAME'] == "United States Virgin Islands"].index)
state_gdf = state_gdf.drop(state_gdf[state_gdf['NAME'] == "American Samoa"].index)
state_gdf = state_gdf.drop(state_gdf[state_gdf['NAME'] == "Guam"].index)
state_gdf = state_gdf.drop(state_gdf[state_gdf['NAME'] == "Puerto Rico"].index)
state_gdf = state_gdf.drop(state_gdf[state_gdf['NAME'] == "Commonwealth of the Northern Mariana Islands"].index)

In [29]:
# Streamline data for use in AI training

new_state_gdf = gpd.GeoDataFrame({})
new_state_gdf['state'] = state_gdf['NAME']
new_state_gdf['land_area'] = state_gdf['ALAND']
new_state_gdf['water_area'] = state_gdf['AWATER']
new_state_gdf['geometry'] = state_gdf['geometry']

new_state_gdf.to_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/state_geometry.shp')

new_state_gdf.head()

C:\Users\EK111\AppData\Local\Temp\ipykernel_55528\3327753784.py:7: FutureWarning: You are adding a column named 'geometry' to a GeoDataFrame constructed without an active geometry column. Currently, this automatically sets the active geometry column to 'geometry' but in the future that will no longer happen. Instead, either provide geometry to the GeoDataFrame constructor (GeoDataFrame(... geometry=GeoSeries()) or use `set_geometry('geometry')` to explicitly set the active geometry column.
  new_state_gdf['geometry'] = state_gdf['geometry']


,state,land_area,water_area,geometry
0,New Mexico,314198519809,726531289,"POLYGON ((-109.05 31.48, -109.05 31.5, -109.05..."
1,South Dakota,196341670967,3387563375,"POLYGON ((-104.06 44.998, -104.05 44.998, -104..."
2,California,403673433805,20291632828,"MULTIPOLYGON (((-118.6 33.479, -118.6 33.478, ..."
3,Kentucky,102266755818,2384136185,"MULTIPOLYGON (((-89.406 36.528, -89.399 36.542..."
4,Alabama,131185561946,4581813708,"MULTIPOLYGON (((-88.053 30.507, -88.051 30.509..."


In [30]:
# Iterate the same geometry information for every year of data analysis except for prediction years

state_geometry_annual = gpd.GeoDataFrame({'state': [], 'land_area': [], 'water_area': [], 'geometry': []})
for i in range(2014, 2025): # from 2014 to 2024; doesn't count the last number 2025
    one_year = new_state_gdf
    one_year['year'] = [i] * len(new_state_gdf)
    state_geometry_annual = state_geometry_annual.merge(one_year, how = 'outer')

state_geometry_annual.head()

,state,land_area,water_area,geometry,year
0,New Mexico,314198519809,726531289,"POLYGON ((-109.05 31.48, -109.05 31.5, -109.05...",2014
1,South Dakota,196341670967,3387563375,"POLYGON ((-104.06 44.998, -104.05 44.998, -104...",2014
2,California,403673433805,20291632828,"MULTIPOLYGON (((-118.6 33.479, -118.6 33.478, ...",2014
3,Kentucky,102266755818,2384136185,"MULTIPOLYGON (((-89.406 36.528, -89.399 36.542...",2014
4,Alabama,131185561946,4581813708,"MULTIPOLYGON (((-88.053 30.507, -88.051 30.509...",2014


In [41]:
# Save annual state data

state_geometry_annual = state_geometry_annual.set_geometry('geometry')
state_geometry_annual.to_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/state_geometry_annual.shp')